<a href="https://colab.research.google.com/github/Sno-7178/Cross-catalyst-estimation-of-Activation-Energy-for-reactions-of-Hydrodeoxygenation-of-propanoic-acid/blob/main/khi_ap_whi_wali_gay_pcawali_toh_nhi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install optuna scikit-learn xgboost lightgbm pandas optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 12.6 MB/s eta 0:00:00


In [ ]:
!pip install rdkit -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 62.1 MB/s eta 0:00:00


In [ ]:
! pip install -U ydata-profiling

KeyboardInterrupt: 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import GroupKFold, cross_val_score, LeaveOneGroupOut

**Loading the data, combining all the features we have evluated**

In [ ]:
df = pd.read_excel('/content/dummy.xlsx').dropna()
print(f"Dataset shape : {df.shape}")
print(f"Catalysts     : {df['Catalyst'].unique()}\n")

Dataset shape : (558, 61)
Catalysts     : ['Cu' 'Ni' 'Pt' 'Rh' 'Pd']



In [ ]:
df.head()

,Catalyst,Reaction,Direction,bond_type,delta_g_rxn,delta_g,pauling_electronegativity,work_function,d_band_center,d_bandwidth,...,sum_product_C-O0,sum_product_C-O1,sum_product_C=O,sum_product_O-H,sum_product_C0-C0,sum_product_C0-C1,sum_product_C0-C2,sum_product_C0-C3,sum_product_C1-C1,sum_product_C1-C2
0,Cu,CH3CH2COOH* + 3* → CH3CH2CO*** + OH*,Forward,OH_removal,0.69,1.41,1.9,4.992,-2.446,0.95,...,0,0,1,1,1,1,0,0,0,0
1,Cu,CH3CH2COOH* + 3* → CH3CHCOOH*** + H*,Forward,dehydrogenation,0.62,1.08,1.9,4.992,-2.446,0.95,...,1,0,1,1,0,2,0,0,0,0
2,Cu,CH3CH2CO*** → CH3CH2* + CO* + *,Forward,decarbonylation,0.09,1.04,1.9,4.992,-2.446,0.95,...,0,0,1,0,0,1,0,0,0,0
3,Cu,CH3CH2CO*** + * → CH3CHCO*** + H*,Forward,dehydrogenation,0.43,1.02,1.9,4.992,-2.446,0.95,...,0,0,1,0,1,1,0,0,0,0
4,Cu,CH3CHCOOH*** + * → CH3CHCO*** + OH*,Forward,OH_removal,0.51,0.95,1.9,4.992,-2.446,0.95,...,0,0,1,1,1,1,0,0,0,0


**All the feature columns we will be using for buildfing the model**

In [ ]:
catalyst_identifier = ['Catalyst']

catalyst_features = ['pauling_electronegativity', 'work_function', 'd_band_center',
                    'd_bandwidth', 'd_band_filling', 'allen_en', 'IP','cohesive','lat_cons']

molecular_descriptors = ['sum_mw', 'sum_logp',
                        'sum_balaban', 'sum_hall']

reaction_features = ['total_stars', 't_n_c',
                    't_n_h', 't_n_o', 'direction', 'bond_type',
                    'delta_g_rxn', 'sum_reactant_O0', 'sum_reactant_O1',
                    'sum_reactant_C0', 'sum_reactant_C1', 'sum_reactant_C2',
                    'sum_reactant_C3', 'sum_reactant_C-H', 'sum_reactant_C=O', 'sum_reactant_O-H',
                    'sum_product_O0', 'sum_product_O1', 'sum_product_C0', 'sum_product_C1',
                    'sum_product_C2', 'sum_product_C3', 'sum_product_C-H',
                    'sum_product_C=O',
                    'sum_product_O-H', ]

label = ['delta_g']

print(f"Number of catalyst features: {len(catalyst_features)}")
print(f"Number of molecular descriptors: {len(molecular_descriptors)}")
print(f"Number of reaction features: {len(reaction_features)}")
print(f"Total number of features: {len(catalyst_features) + len(molecular_descriptors) + len(reaction_features)}")

Number of catalyst features: 9
Number of molecular descriptors: 4
Number of reaction features: 25
Total number of features: 38


**No. of each bond types**

In [ ]:
print(f"Total number of bond types: {len(df["bond_type"].unique())}")
print(f"\n═══════════════════════════════ BOND TYPES ═══════════════════════════════\n")
print(f"{'Bond Type: ':<30} {'Occurrences: ':<10}\n")
for r in df["bond_type"].unique():
  print(f"{r:<30} {len(df[df["bond_type"]==r]):<10}")

Total number of bond types: 16

═══════════════════════════════ BOND TYPES ═══════════════════════════════

Bond Type:                     Occurrences: 

OH_removal                     30        
dehydrogenation                175       
decarbonylation                25        
hydrogenation                  176       
decarboxylation                14        
CC_scission                    18        
carboxyl_dissociation          10        
water_dissociation             5         
H2_dissociation                1         
CC_formation                   37        
OH_addition                    30        
carboxyl_formation             22        
water_formation                5         
H2_association                 2         
carbonylation                  5         
carboxylation                  3         


**Performing feature enginnering on some of the columns:-
* Encoding the **Direction** and **Bond Type** columns
* Keeping other columns the same

In [ ]:
# One-hot encoding of categorical columns
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_enc = ohe.fit_transform(df[['bond_type','Direction']])
cat_cols = ohe.get_feature_names_out(['bond_type','Direction'])
df_cat  = pd.DataFrame(cat_enc, columns=cat_cols, index=df.index)

# Reaction feature columns
RF_COLS_CANDIDATE = ['total_stars', 't_n_c', 'sum_reactant_O0','sum_product_O0', 'sum_reactant_O1','sum_product_O1',
                    'sum_reactant_C0',  'sum_product_C0', 'sum_reactant_C1',  'sum_product_C1','sum_reactant_C2',
                      'sum_product_C2','sum_reactant_C-H', 'sum_product_C-H',
                     'sum_reactant_C=O', 'sum_product_C=O','sum_reactant_O-H','sum_product_O-H']

rf_cols = [c for c in RF_COLS_CANDIDATE if c in df.columns]
# Catalyst columns
cat_feats = ['d_band_center','d_bandwidth', 'd_band_filling', 'pauling_electronegativity', 'allen_en','work_function',
             'IP','cohesive','lat_cons']
# descriptor columns
mol_descps = ['sum_mw', 'sum_logp',
                        'sum_balaban', 'sum_hall']
# Numerical columns
NUMERIC_COLS =cat_feats + mol_descps + ["delta_g_rxn"]
num_cols = [c for c in NUMERIC_COLS if c in df.columns]

In [ ]:
X = pd.concat([df[num_cols], df[rf_cols], df_cat], axis=1).fillna(0).astype(float) # DataFrame
y = df[label].astype(float) # Series
print(f"Feature matrix : {X.shape}")
print(f"Target shape   : {y.shape}\n")

Feature matrix : (558, 50)
Target shape   : (558, 1)



In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="EDA Report")
profile.to_file("eda_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 61/61 [00:01<00:00, 48.90it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from sklearn.decomposition import PCA



# ══════════════ **RANDOM FOREST** ══════════════




In [ ]:
groups = df["Catalyst"].values
groups

array(['Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu',
       'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'Cu', 'C

In [ ]:
import pandas as pd

# ── 1. Optuna objective with scaling inside each trial ────────────────────────
def objective(trial, X, y, groups):
    model = RandomForestRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 100, 300),
        max_depth        = trial.suggest_int('max_depth', 3, 12),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 10),
        max_features     = trial.suggest_float('max_features', 0.3, 0.9),
        random_state     = 42,
        n_jobs           = -1
    )

    cv     = LeaveOneGroupOut()
    scores = []

    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        # Fit scaler ONLY on training fold — transform both
        scaler = StandardScaler()

        # Scale the numerical features
        X_tr_scaled = scaler.fit_transform(X_tr[num_cols])
        X_val_scaled = scaler.transform(X_val[num_cols])

        # Apply PCA
        X_tr_pca = pca.fit_transform(X_tr_scaled)
        X_val_pca = pca.transform(X_val_scaled)

        # Drop the original numerical columns from X_tr and X_val
        #X_tr = X_tr.drop(columns=num_cols)
        #X_val = X_val.drop(columns=num_cols)



        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))

    return np.mean(scores)


In [ ]:
# ── 2. Run Optuna ─────────────────────────────────────────────────────────────
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective(trial, X, y, groups),
    n_trials          = 80,
    show_progress_bar = True
)
rf_best_params = study.best_params
print("Best params :", rf_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

  0%|          | 0/80 [00:00<?, ?it/s]

Best params : {'n_estimators': 223, 'max_depth': 12, 'min_samples_leaf': 2, 'max_features': 0.6946363753274336}
Best CV R²  : 0.6591


In [ ]:
# ── 3. Final LOCO evaluation with scaling inside each fold ────────────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

print("═════════════════════════════════════ RANDOM FOREST ═════════════════════════════════════\n")
for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    # Fit scaler ONLY on training catalysts — transform test separately
    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = RandomForestRegressor(**rf_best_params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")


# ── 4. Summary ────────────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")

═════════════════════════════════════ RANDOM FOREST ═════════════════════════════════════

Held-out: Cu              R²=0.6060  MAE=0.1718  RMSE=0.2381  MAPE=57.46%
Held-out: Ni              R²=0.5766  MAE=0.2071  RMSE=0.2500  MAPE=inf%
Held-out: Pd              R²=0.7541  MAE=0.1314  RMSE=0.1824  MAPE=44.07%
Held-out: Pt              R²=0.6265  MAE=0.1993  RMSE=0.2600  MAPE=110.45%
Held-out: Rh              R²=0.7298  MAE=0.1483  RMSE=0.1882  MAPE=inf%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6586     0.0703     0.5766     0.7541
MAE          0.1716     0.0289     0.1314     0.2071
RMSE         0.2237     0.0322     0.1824     0.2600
MSE          0.0511     0.0141     0.0333     0.0676
MAPE%           inf        nan      44.07        inf


In [ ]:
import matplotlib.pyplot as plt

# ── 1. Optuna objective with scaling inside each trial ────────────────────────
def objective(trial, X, y, groups):
    model = RandomForestRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 100, 300),
        max_depth        = trial.suggest_int('max_depth', 6, 12),
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 10),
        max_features     = trial.suggest_float('max_features', 0.3, 0.9),
        random_state     = 42,
        n_jobs           = -1
    )

    cv     = LeaveOneGroupOut()
    scores = []

    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        scaler = StandardScaler()
        X_tr[num_cols]  = scaler.fit_transform(X_tr[num_cols])
        X_val[num_cols] = scaler.transform(X_val[num_cols])

        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))

    return np.mean(scores)

# ── 2. Run Optuna ─────────────────────────────────────────────────────────────
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective(trial, X, y, groups),
    n_trials          = 50,
    show_progress_bar = True
)
rf_best_params = study.best_params
print("Best params :", rf_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

# ── 3. Final LOCO evaluation with scaling inside each fold ────────────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

# Storage for both importance types across folds
mdi_records  = []   # built-in MDI importances  — one array per fold
perm_records = []   # permutation importances   — one array per fold

print("═" * 80)
print("RANDOM FOREST — LOCO")
print("═" * 80 + "\n")

from sklearn.inspection import permutation_importance

for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = RandomForestRegressor(**rf_best_params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds  = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  "
          f"RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    # ── MDI (built-in impurity) importance — from training fold ───────────────
    mdi_records.append(model.feature_importances_)

    # ── Permutation importance — evaluated on the held-out test fold ──────────
    # n_repeats=10 gives stable estimates; random_state for reproducibility
    perm = permutation_importance(
        model, X_test, y_true,
        n_repeats=10,
        scoring='r2',
        random_state=42,
        n_jobs=-1
    )
    perm_records.append(perm.importances_mean)

# ── 4. Summary metrics ────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")

# ══════════════════════════════════════════════════════════════════════════════
# ── 5. Feature Importance — aggregated across all LOCO folds ─────────────────
# ══════════════════════════════════════════════════════════════════════════════

feature_names = X.columns.tolist()

# Mean + std across folds for both importance types
mdi_mean  = np.mean(mdi_records,  axis=0)
mdi_std   = np.std(mdi_records,   axis=0)
perm_mean = np.mean(perm_records, axis=0)
perm_std  = np.std(perm_records,  axis=0)

# Build summary DataFrame sorted by MDI importance
df_importance = pd.DataFrame({
    'Feature':         feature_names,
    'MDI_mean':        mdi_mean,
    'MDI_std':         mdi_std,
    'Perm_mean':       perm_mean,
    'Perm_std':        perm_std,
}).sort_values('MDI_mean', ascending=False).reset_index(drop=True)

print("\n" + "═" * 70)
print("FEATURE IMPORTANCE — averaged across all LOCO folds")
print("═" * 70)
print(df_importance.to_string(index=False))

# ── 6. Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(feature_names) * 0.35)))

# Sort separately for each plot
mdi_order  = df_importance.sort_values('MDI_mean',  ascending=True)
perm_order = df_importance.sort_values('Perm_mean', ascending=True)

# ── MDI plot ──────────────────────────────────────────────────────────────────
axes[0].barh(
    mdi_order['Feature'], mdi_order['MDI_mean'],
    xerr=mdi_order['MDI_std'],
    color='steelblue', alpha=0.85, capsize=3, ecolor='navy'
)
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Mean MDI Importance (± std across folds)', fontsize=11)
axes[0].set_title('MDI Feature Importance\n(Built-in, averaged over LOCO folds)', fontsize=12)
axes[0].tick_params(axis='y', labelsize=9)

# ── Permutation importance plot ───────────────────────────────────────────────
axes[1].barh(
    perm_order['Feature'], perm_order['Perm_mean'],
    xerr=perm_order['Perm_std'],
    color='darkorange', alpha=0.85, capsize=3, ecolor='saddlebrown'
)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Mean Permutation Importance (R² drop, ± std across folds)', fontsize=11)
axes[1].set_title('Permutation Feature Importance\n(Evaluated on held-out catalyst, averaged over LOCO folds)', fontsize=12)
axes[1].tick_params(axis='y', labelsize=9)

plt.suptitle('Random Forest — Feature Importance across LOCO folds', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅  Plot saved → rf_feature_importance.png")


  0%|          | 0/50 [00:00<?, ?it/s]

Best params : {'n_estimators': 289, 'max_depth': 10, 'min_samples_leaf': 2, 'max_features': 0.8254990520497608}
Best CV R²  : 0.6579
════════════════════════════════════════════════════════════════════════════════
RANDOM FOREST — LOCO
════════════════════════════════════════════════════════════════════════════════

Held-out: Cu              R²=0.6143  MAE=0.1686  RMSE=0.2356  MAPE=57.77%
Held-out: Ni              R²=0.5765  MAE=0.2080  RMSE=0.2501  MAPE=inf%
Held-out: Pd              R²=0.7546  MAE=0.1309  RMSE=0.1822  MAPE=44.16%
Held-out: Pt              R²=0.6169  MAE=0.2025  RMSE=0.2633  MAPE=110.45%
Held-out: Rh              R²=0.7271  MAE=0.1491  RMSE=0.1891  MAPE=inf%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6579     0.0698     0.5765     0.7546
MAE          0.1718     0.0298     0.1309     0.2080
RMSE         0.2241     0.0326     0.1822     0.2633
MSE          0.0513     0.0144     0.0332     0.0693

NameError: name 'plt' is not defined

# ══════════════ **XGBOOST** ══════════════

**Keeping the columns as it is**

In [ ]:
# ── 1. Optuna objective with scaling inside each trial ────────────────────────
def objective(trial, X, y, groups):
    model = XGBRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 400, 500),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        max_depth        = trial.suggest_int('max_depth', 3, 8),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda       = trial.suggest_float('reg_lambda', 1, 10),
        random_state=42, verbosity=0,
    )

    cv     = LeaveOneGroupOut()
    scores = []

    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        # Fit scaler ONLY on training fold — transform both
        scaler = StandardScaler()
        X_tr[num_cols]= scaler.fit_transform(X_tr[num_cols])
        X_val[num_cols] = scaler.transform(X_val[num_cols])
        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))

    return np.mean(scores)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# ✅ FIXED make_objective
# model_fn(trial) takes a function, not a built model
# ColumnTransformer is instantiated fresh per fold — no shared state

def make_objective(model_fn, X, y, groups, num_cols):   # <-- model_fn, not model
    def objective(trial):
        model = model_fn(trial)          # <-- built HERE using trial's suggestions
        cv = LeaveOneGroupOut()
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            pipe = Pipeline([
                ('pre',   ColumnTransformer(                 # <-- fresh instance per fold
                              [('scale', StandardScaler(), num_cols)],
                              remainder='passthrough')),
                ('model', clone(model)),                     # <-- clone per fold
            ])
            pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
            scores.append(r2_score(y.iloc[val_idx], pipe.predict(X.iloc[val_idx])))
        return np.mean(scores)
    return objective

# ✅ model_fn — a function that takes trial and returns a model
def xgb_model(trial):
    return XGBRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 400, 500),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        max_depth        = trial.suggest_int('max_depth', 3, 8),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda       = trial.suggest_float('reg_lambda', 1, 10),
        random_state=42, verbosity=0,
    )

# ✅ Usage — pass xgb_model (the function), not xgb_model(trial)
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(make_objective(xgb_model, X, y, groups, num_cols), n_trials=80,
               show_progress_bar=True)
print(f"Best R²: {study.best_value:.4f}  params: {study.best_params}")

  0%|          | 0/80 [00:00<?, ?it/s]

Best R²: 0.6723  params: {'n_estimators': 493, 'learning_rate': 0.033761884000319986, 'max_depth': 4, 'subsample': 0.5480404041496569, 'colsample_bytree': 0.938874840173581, 'reg_lambda': 4.681009639632912}


In [ ]:
ct = ColumnTransformer([('scale', StandardScaler(), num_cols)], remainder='passthrough')
out = ct.fit_transform(X)
print(out.shape)
print(ct.get_feature_names_out())  # check the column order

(558, 50)
['scale__total_stars' 'scale__t_n_c' 'scale__sum_reactant_O0'
 'scale__sum_product_O0' 'scale__sum_reactant_O1' 'scale__sum_product_O1'
 'scale__sum_reactant_C0' 'scale__sum_product_C0' 'scale__sum_reactant_C1'
 'scale__sum_product_C1' 'scale__sum_reactant_C2' 'scale__sum_product_C2'
 'scale__sum_reactant_C-H' 'scale__sum_product_C-H'
 'scale__sum_reactant_C=O' 'scale__sum_product_C=O'
 'scale__sum_reactant_O-H' 'scale__sum_product_O-H' 'scale__d_band_center'
 'scale__d_bandwidth' 'scale__d_band_filling'
 'scale__pauling_electronegativity' 'scale__allen_en'
 'scale__work_function' 'scale__IP' 'scale__cohesive' 'scale__lat_cons'
 'scale__sum_mw' 'scale__sum_logp' 'scale__sum_balaban' 'scale__sum_hall'
 'scale__delta_g_rxn' 'remainder__bond_type_CC_formation'
 'remainder__bond_type_CC_scission' 'remainder__bond_type_H2_association'
 'remainder__bond_type_H2_dissociation' 'remainder__bond_type_OH_addition'
 'remainder__bond_type_OH_removal' 'remainder__bond_type_carbonylation'
 

**PCA transformed the columns**

In [ ]:
def objective(trial, X, y, groups):
    model = XGBRegressor(
        n_estimators     = trial.suggest_int('n_estimators', 400, 500),
        learning_rate    = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        max_depth        = trial.suggest_int('max_depth', 3, 8),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda       = trial.suggest_float('reg_lambda', 1, 10),
        random_state=42, verbosity=0,
    )

    pca_cat  = PCA(n_components=4)   # catalyst properties
    pca_mol  = PCA(n_components=4)   # molecular descriptors
    pca_rf   = PCA(n_components=12)  # reactant + product atom count

    cv     = LeaveOneGroupOut()
    scores = []
    printed = [False]
    #def print_cumvar(label, pca_obj):
    #    cumvar = np.cumsum(pca_obj.explained_variance_ratio_)
    #    print(f"\n── {label} ({pca_obj.n_components_} components) ──")
    #    for i, v in enumerate(cumvar, 1):
    #        marker = " ◄ 90%" if abs(v - 0.90) == min(abs(cumvar - 0.90)) else \
    #                 " ◄ 95%" if abs(v - 0.97) == min(abs(cumvar - 0.97)) else ""
    #        print(f"  {i:>3} components → {v:.4f} ({v*100:.1f}%){marker}")

    def scale_and_pca(pca_obj, cols, X_tr, X_val):
        scaler = StandardScaler()
        tr_scaled  = scaler.fit_transform(X_tr[cols])
        val_scaled = scaler.transform(X_val[cols])

        tr_pca  = pca_obj.fit_transform(tr_scaled)
        val_pca = pca_obj.transform(val_scaled)

        prefix   = [f'{pca_obj}_pca_{i}' for i in range(pca_obj.n_components_)]
        tr_df  = pd.DataFrame(tr_pca,  columns=prefix, index=X_tr.index)
        val_df = pd.DataFrame(val_pca, columns=prefix, index=X_val.index)
        return tr_df, val_df

    # ── column name prefixes for PCA output DataFrames ───────────────────────

    GROUP_CFG = [
        ('cat',  pca_cat,  cat_feats),
        ('mol',  pca_mol,  mol_descps),
        ('rf',   pca_rf,   rf_cols),
    ]

    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        all_num_cols = cat_feats + mol_descps + rf_cols

        pca_tr_parts  = []
        pca_val_parts = []

        for prefix, pca_obj, cols in GROUP_CFG:
            scaler     = StandardScaler()
            tr_scaled  = scaler.fit_transform(X_tr[cols])
            val_scaled = scaler.transform(X_val[cols])

            tr_pca  = pca_obj.fit_transform(tr_scaled)
            val_pca = pca_obj.transform(val_scaled)

            pca_col_names = [f'{prefix}_pc{i}' for i in range(pca_obj.n_components_)]
            pca_tr_parts.append(
                pd.DataFrame(tr_pca,  columns=pca_col_names, index=X_tr.index))
            pca_val_parts.append(
                pd.DataFrame(val_pca, columns=pca_col_names, index=X_val.index))

        # ── diagnostics: print once across all trials/folds ──────────────────
        #if not printed[0]:
        #    print(f"X_tr shape (before PCA): {X_tr.shape}")
        #    for prefix, pca_obj, cols in GROUP_CFG:
        #        print_cumvar(f"{prefix.upper()} group  |  source cols: {len(cols)}", pca_obj)
        #    printed[0] = True

        # ── assemble final feature matrix ─────────────────────────────────────
        other_cols = X_tr.drop(columns=all_num_cols)   # encoded categoricals etc.

        X_tr_processed  = pd.concat([other_cols,
                                     *pca_tr_parts],  axis=1)
        X_val_processed = pd.concat([X_val.drop(columns=all_num_cols),
                                     *pca_val_parts], axis=1)

        #if not printed[0]:   # shape printed right after first assembly
        #    print(f"\nX_tr_processed shape (after PCA): {X_tr_processed.shape}")

        model.fit(X_tr_processed, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val_processed)))

    return np.mean(scores)

In [ ]:
groups = df["Catalyst"].values

# ── 2. Run Optuna ─────────────────────────────────────────────────────────────
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective(trial, X, y, groups),
    n_trials          = 80,
    show_progress_bar = True
)
xgb_best_params = study.best_params
print("Best params :", xgb_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

  0%|          | 0/80 [00:00<?, ?it/s]

Best params : {'n_estimators': 467, 'learning_rate': 0.03339274980152531, 'max_depth': 4, 'subsample': 0.6487916050461701, 'colsample_bytree': 0.5999795965653543, 'reg_lambda': 1.6163146351252202}
Best CV R²  : 0.6741


In [ ]:
# ── 3. Final LOCO evaluation with scaling inside each fold ────────────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

print("═════════════════════════════════════ XGBOOST ═════════════════════════════════════\n")
for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    # Fit scaler ONLY on training catalysts — transform test separately
    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = XGBRegressor(**xgb_best_params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")


# ── 4. Summary ────────────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")

═════════════════════════════════════ XGBOOST ═════════════════════════════════════

Held-out: Cu              R²=0.5992  MAE=0.1683  RMSE=0.2402  MAPE=56.54%
Held-out: Ni              R²=0.6556  MAE=0.1946  RMSE=0.2255  MAPE=inf%
Held-out: Pd              R²=0.8131  MAE=0.1141  RMSE=0.1590  MAPE=45.27%
Held-out: Pt              R²=0.7062  MAE=0.1768  RMSE=0.2306  MAPE=117.15%
Held-out: Rh              R²=0.7184  MAE=0.1531  RMSE=0.1922  MAPE=inf%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6985     0.0711     0.5992     0.8131
MAE          0.1614     0.0272     0.1141     0.1946
RMSE         0.2095     0.0300     0.1590     0.2402
MSE          0.0448     0.0120     0.0253     0.0577
MAPE%           inf        nan      45.27        inf


# ══════════════ **LIGHT GBM** ══════════════

In [ ]:
# ── 1. Optuna objective with scaling inside each trial ────────────────────────
def objective(trial, X, y, groups):
    model = LGBMRegressor(
          n_estimators      = trial.suggest_int('n_estimators', 100, 500),
          learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
          num_leaves        = trial.suggest_int('num_leaves', 20, 80),
          max_depth         = trial.suggest_int('max_depth', 3, 10),
          min_child_samples = trial.suggest_int('min_child_samples', 5, 50),
          subsample         = trial.suggest_float('subsample', 0.5, 1.0),
          colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
          reg_lambda        = trial.suggest_float('reg_lambda', 1, 10),
          random_state=42, verbose=-1,
    )

    cv     = LeaveOneGroupOut()
    scores = []

    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        # Fit scaler ONLY on training fold — transform both
        scaler = StandardScaler()

        X_tr[num_cols]  = scaler.fit_transform(X_tr[num_cols])
        X_val[num_cols] = scaler.transform(X_val[num_cols])

        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))

    return np.mean(scores)


In [ ]:
  def objective(trial, X, y, groups):
    model = LGBMRegressor(
          n_estimators      = trial.suggest_int('n_estimators', 100, 500),
          learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
          num_leaves        = trial.suggest_int('num_leaves', 20, 80),
          max_depth         = trial.suggest_int('max_depth', 3, 10),
          min_child_samples = trial.suggest_int('min_child_samples', 5, 50),
          subsample         = trial.suggest_float('subsample', 0.5, 1.0),
          colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
          reg_lambda        = trial.suggest_float('reg_lambda', 1, 10),
          random_state=42, verbose=-1,
    )

    cv     = LeaveOneGroupOut()
    scores = []
    pca_cat  = PCA(n_components=4)   # catalyst properties
    pca_mol  = PCA(n_components=4)   # molecular descriptors
    pca_rf   = PCA(n_components=12)   # reactant + product atom count

    cv     = LeaveOneGroupOut()
    scores = []
    printed = [False]
    #def print_cumvar(label, pca_obj):
    #    cumvar = np.cumsum(pca_obj.explained_variance_ratio_)
    #    print(f"\n── {label} ({pca_obj.n_components_} components) ──")
    #    for i, v in enumerate(cumvar, 1):
    #        marker = " ◄ 90%" if abs(v - 0.90) == min(abs(cumvar - 0.90)) else \
    #                 " ◄ 95%" if abs(v - 0.97) == min(abs(cumvar - 0.97)) else ""
    #        print(f"  {i:>3} components → {v:.4f} ({v*100:.1f}%){marker}")

    def scale_and_pca(pca_obj, cols, X_tr, X_val):
        scaler = StandardScaler()
        tr_scaled  = scaler.fit_transform(X_tr[cols])
        val_scaled = scaler.transform(X_val[cols])

        tr_pca  = pca_obj.fit_transform(tr_scaled)
        val_pca = pca_obj.transform(val_scaled)

        prefix   = [f'{pca_obj}_pca_{i}' for i in range(pca_obj.n_components_)]
        tr_df  = pd.DataFrame(tr_pca,  columns=prefix, index=X_tr.index)
        val_df = pd.DataFrame(val_pca, columns=prefix, index=X_val.index)
        return tr_df, val_df

    # ── column name prefixes for PCA output DataFrames ───────────────────────

    GROUP_CFG = [
        ('cat',  pca_cat,  cat_feats),
        ('mol',  pca_mol,  mol_descps),
        ('rf',   pca_rf,   rf_cols),
    ]

    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        all_num_cols = cat_feats + mol_descps + rf_cols

        pca_tr_parts  = []
        pca_val_parts = []

        for prefix, pca_obj, cols in GROUP_CFG:
            scaler     = StandardScaler()
            tr_scaled  = scaler.fit_transform(X_tr[cols])
            val_scaled = scaler.transform(X_val[cols])

            tr_pca  = pca_obj.fit_transform(tr_scaled)
            val_pca = pca_obj.transform(val_scaled)

            pca_col_names = [f'{prefix}_pc{i}' for i in range(pca_obj.n_components_)]
            pca_tr_parts.append(
                pd.DataFrame(tr_pca,  columns=pca_col_names, index=X_tr.index))
            pca_val_parts.append(
                pd.DataFrame(val_pca, columns=pca_col_names, index=X_val.index))

        # ── diagnostics: print once across all trials/folds ──────────────────
        #if not printed[0]:
        #    print(f"X_tr shape (before PCA): {X_tr.shape}")
        #    for prefix, pca_obj, cols in GROUP_CFG:
        #        print_cumvar(f"{prefix.upper()} group  |  source cols: {len(cols)}", pca_obj)
        #    printed[0] = True

        # ── assemble final feature matrix ─────────────────────────────────────
        other_cols = X_tr.drop(columns=all_num_cols)   # encoded categoricals etc.

        X_tr_processed  = pd.concat([other_cols,
                                     *pca_tr_parts],  axis=1)
        X_val_processed = pd.concat([X_val.drop(columns=all_num_cols),
                                     *pca_val_parts], axis=1)

        #if not printed[0]:   # shape printed right after first assembly
        #    print(f"\nX_tr_processed shape (after PCA): {X_tr_processed.shape}")

        model.fit(X_tr_processed, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val_processed)))

    return np.mean(scores)

In [ ]:
# ── 2. Run Optuna ─────────────────────────────────────────────────────────────
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective(trial, X, y, groups),
    n_trials          = 80,
    show_progress_bar = True
)
lgbm_best_params = study.best_params
print("Best params :", lgbm_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

  0%|          | 0/80 [00:00<?, ?it/s]

Best params : {'n_estimators': 494, 'learning_rate': 0.03828540040052391, 'num_leaves': 42, 'max_depth': 5, 'min_child_samples': 15, 'subsample': 0.5817147621039088, 'colsample_bytree': 0.9486511514003211, 'reg_lambda': 6.055108643204481}
Best CV R²  : 0.6957


In [ ]:
# ── 3. Final LOCO evaluation with scaling inside each fold ────────────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

print("═════════════════════════════════════ LightGBM ═════════════════════════════════════\n")
for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    # Fit scaler ONLY on training catalysts — transform test separately
    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = LGBMRegressor(**lgbm_best_params, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")


# ── 4. Summary ────────────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")

═════════════════════════════════════ LightGBM ═════════════════════════════════════

Held-out: Cu              R²=0.5832  MAE=0.1749  RMSE=0.2449  MAPE=57.21%
Held-out: Ni              R²=0.6417  MAE=0.1934  RMSE=0.2300  MAPE=inf%
Held-out: Pd              R²=0.8219  MAE=0.1248  RMSE=0.1552  MAPE=47.89%
Held-out: Pt              R²=0.7091  MAE=0.1799  RMSE=0.2294  MAPE=114.09%
Held-out: Rh              R²=0.7253  MAE=0.1556  RMSE=0.1898  MAPE=inf%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6962     0.0807     0.5832     0.8219
MAE          0.1657     0.0238     0.1248     0.1934
RMSE         0.2099     0.0329     0.1552     0.2449
MSE          0.0451     0.0131     0.0241     0.0600
MAPE%           inf        nan      47.89        inf


# ══════════════ **SVR** ══════════════

In [ ]:
from sklearn.svm import SVR

# ── 1. Optuna objective ───────────────────────────────────────────────────────
def objective(trial, X, y, groups):
    model = SVR(
        C       = trial.suggest_float('C', 0.1, 100, log=True),
        epsilon = trial.suggest_float('epsilon', 0.01, 1.0, log=True),
        gamma   = trial.suggest_categorical('gamma', ['scale', 'auto']),
        kernel  = trial.suggest_categorical('kernel', ['rbf', 'linear', 'poly']),
    )
    cv     = LeaveOneGroupOut()
    scores = []
    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        scaler = RobustScaler()

        X_tr[num_cols]  = scaler.fit_transform(X_tr[num_cols])
        X_val[num_cols] = scaler.transform(X_val[num_cols])

        model.fit(X_tr, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val)))
    return np.mean(scores)


# ── 2. Run Optuna ─────────────────────────────────────────────────────────────
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective(trial, X, y, groups),
    n_trials          = 50,
    show_progress_bar = True
)
svr_best_params = study.best_params
print("Best params :", svr_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")


# ── 3. Final LOCO evaluation ──────────────────────────────────────────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

print("══════════════════════════════════════ SVR ══════════════════════════════════════\n")

for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    scaler = RobustScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = SVR(**svr_best_params)
    model.fit(X_train, y_train)
    preds  = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")


# ── 4. Summary ────────────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
from sklearn.svm import SVR

# ── 1. Optuna objective ───────────────────────────────────────────────────────
def objective(trial, X, y, groups):
    model = SVR(
        C       = trial.suggest_float('C', 0.1, 100, log=True),
        epsilon = trial.suggest_float('epsilon', 0.01, 1.0, log=True),
        gamma   = trial.suggest_categorical('gamma', ['scale', 'auto']),
        kernel  = trial.suggest_categorical('kernel', ['rbf', 'linear', 'poly']),
    )
    cv     = LeaveOneGroupOut()
    scores = []
    pca_cat  = PCA(n_components=4)   # catalyst properties
    pca_mol  = PCA(n_components=4)   # molecular descriptors
    pca_rf   = PCA(n_components=12)   # reactant + product atom count

    cv     = LeaveOneGroupOut()
    scores = []
    printed = [False]
    #def print_cumvar(label, pca_obj):
    #    cumvar = np.cumsum(pca_obj.explained_variance_ratio_)
    #    print(f"\n── {label} ({pca_obj.n_components_} components) ──")
    #    for i, v in enumerate(cumvar, 1):
    #        marker = " ◄ 90%" if abs(v - 0.90) == min(abs(cumvar - 0.90)) else \
    #                 " ◄ 95%" if abs(v - 0.97) == min(abs(cumvar - 0.97)) else ""
    #        print(f"  {i:>3} components → {v:.4f} ({v*100:.1f}%){marker}")

    def scale_and_pca(pca_obj, cols, X_tr, X_val):
        scaler = StandardScaler()
        tr_scaled  = scaler.fit_transform(X_tr[cols])
        val_scaled = scaler.transform(X_val[cols])

        tr_pca  = pca_obj.fit_transform(tr_scaled)
        val_pca = pca_obj.transform(val_scaled)

        prefix   = [f'{pca_obj}_pca_{i}' for i in range(pca_obj.n_components_)]
        tr_df  = pd.DataFrame(tr_pca,  columns=prefix, index=X_tr.index)
        val_df = pd.DataFrame(val_pca, columns=prefix, index=X_val.index)
        return tr_df, val_df

    # ── column name prefixes for PCA output DataFrames ───────────────────────

    GROUP_CFG = [
        ('cat',  pca_cat,  cat_feats),
        ('mol',  pca_mol,  mol_descps),
        ('rf',   pca_rf,   rf_cols),
    ]

    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        all_num_cols = cat_feats + mol_descps + rf_cols

        pca_tr_parts  = []
        pca_val_parts = []

        for prefix, pca_obj, cols in GROUP_CFG:
            scaler     = StandardScaler()
            tr_scaled  = scaler.fit_transform(X_tr[cols])
            val_scaled = scaler.transform(X_val[cols])

            tr_pca  = pca_obj.fit_transform(tr_scaled)
            val_pca = pca_obj.transform(val_scaled)

            pca_col_names = [f'{prefix}_pc{i}' for i in range(pca_obj.n_components_)]
            pca_tr_parts.append(
                pd.DataFrame(tr_pca,  columns=pca_col_names, index=X_tr.index))
            pca_val_parts.append(
                pd.DataFrame(val_pca, columns=pca_col_names, index=X_val.index))

        # ── diagnostics: print once across all trials/folds ──────────────────
        #if not printed[0]:
        #    print(f"X_tr shape (before PCA): {X_tr.shape}")
        #    for prefix, pca_obj, cols in GROUP_CFG:
        #        print_cumvar(f"{prefix.upper()} group  |  source cols: {len(cols)}", pca_obj)
        #    printed[0] = True

        # ── assemble final feature matrix ─────────────────────────────────────
        other_cols = X_tr.drop(columns=all_num_cols)   # encoded categoricals etc.

        X_tr_processed  = pd.concat([other_cols,
                                     *pca_tr_parts],  axis=1)
        X_val_processed = pd.concat([X_val.drop(columns=all_num_cols),
                                     *pca_val_parts], axis=1)

        #if not printed[0]:   # shape printed right after first assembly
        #    print(f"\nX_tr_processed shape (after PCA): {X_tr_processed.shape}")

        model.fit(X_tr_processed, y_tr)
        scores.append(r2_score(y_val, model.predict(X_val_processed)))

    return np.mean(scores)

In [ ]:
# ── 2. Run Optuna ─────────────────────────────────────────────────────────────
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective(trial, X, y, groups),
    n_trials          = 80,
    show_progress_bar = True
)
svr_best_params = study.best_params
print("Best params :", svr_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")

  0%|          | 0/80 [00:00<?, ?it/s]

Best params : {'C': 4.248779976521729, 'epsilon': 0.024785671646681568, 'gamma': 'auto', 'kernel': 'rbf'}
Best CV R²  : 0.6454


# ══════════════ **GPR** ══════════════

In [ ]:
# ── 3. Final LOCO evaluation with scaling inside each fold ────────────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

GPR_KERNEL = (
    C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5)
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
)

print("═════════════════════════════════════ GPR ═════════════════════════════════════\n")
for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    # Fit scaler ONLY on training catalysts — transform test separately
    scaler = StandardScaler()

    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    model = GaussianProcessRegressor(
                             kernel=GPR_KERNEL, alpha=1e-6,
                             normalize_y=True, n_restarts_optimizer=5,
                             random_state=42)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")


# ── 4. Summary ────────────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")

═════════════════════════════════════ GPR ═════════════════════════════════════

Held-out: Cu              R²=0.5698  MAE=0.1848  RMSE=0.2488  MAPE=59.89%
Held-out: Ni              R²=0.6252  MAE=0.2004  RMSE=0.2352  MAPE=inf%
Held-out: Pd              R²=0.8089  MAE=0.1146  RMSE=0.1608  MAPE=44.60%
Held-out: Pt              R²=0.6792  MAE=0.1816  RMSE=0.2409  MAPE=116.45%
Held-out: Rh              R²=0.7443  MAE=0.1438  RMSE=0.1831  MAPE=inf%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6855     0.0845     0.5698     0.8089
MAE          0.1650     0.0313     0.1146     0.2004
RMSE         0.2138     0.0351     0.1608     0.2488
MSE          0.0469     0.0144     0.0259     0.0619
MAPE%           inf        nan      44.60        inf


In [ ]:
# ── 1. Optuna objective with PCA inside each trial ────────────────────────
def objective(trial, X, y, groups):
    # Tunable parameters for GPR (e.g., n_restarts_optimizer)
    n_restarts_optimizer = trial.suggest_int('n_restarts_optimizer', 0, 10)

    model = GaussianProcessRegressor(
        kernel=GPR_KERNEL,
        alpha=1e-6,
        normalize_y=True,
        n_restarts_optimizer=n_restarts_optimizer,
        random_state=42
    )

    pca_cat  = PCA(n_components=4)   # catalyst properties
    pca_mol  = PCA(n_components=4)   # molecular descriptors
    pca_rf   = PCA(n_components=12)   # reactant + product atom count

    GROUP_CFG = [
        ('cat',  pca_cat,  cat_feats),
        ('mol',  pca_mol,  mol_descps),
        ('rf',   pca_rf,   rf_cols),
    ]
    all_num_cols = cat_feats + mol_descps + rf_cols

    cv     = LeaveOneGroupOut()
    scores = []

    for train_idx, val_idx in cv.split(X, y, groups):
        X_tr  = X.iloc[train_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        pca_tr_parts  = []
        pca_val_parts = []

        for prefix, pca_obj, cols in GROUP_CFG:
            scaler     = StandardScaler()
            tr_scaled  = scaler.fit_transform(X_tr[cols])
            val_scaled = scaler.transform(X_val[cols])

            tr_pca  = pca_obj.fit_transform(tr_scaled)
            val_pca = pca_obj.transform(val_scaled)

            pca_col_names = [f'{prefix}_pc{i}' for i in range(pca_obj.n_components_)]
            pca_tr_parts.append(
                pd.DataFrame(tr_pca,  columns=pca_col_names, index=X_tr.index))
            pca_val_parts.append(
                pd.DataFrame(val_pca, columns=pca_col_names, index=X_val.index))

        other_cols = X_tr.drop(columns=all_num_cols, errors='ignore')
        X_tr_processed  = pd.concat([other_cols, *pca_tr_parts], axis=1)

        other_cols_val = X_val.drop(columns=all_num_cols, errors='ignore')
        X_val_processed = pd.concat([other_cols_val, *pca_val_parts], axis=1)

        model.fit(X_tr_processed, y_tr.values.ravel())
        scores.append(r2_score(y_val, model.predict(X_val_processed)))

    return np.mean(scores)

# GPR_KERNEL needs to be defined before the objective function is called
GPR_KERNEL = (
    C(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=2.5)
    + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-5, 1e1))
)

# ── 2. Run Optuna ─────────────────────────────────────────────────────────────
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(
    lambda trial: objective(trial, X, y, groups),
    n_trials          = 80,
    show_progress_bar = True
)
gpr_best_params = study.best_params
print("Best params :", gpr_best_params)
print(f"Best CV R²  : {study.best_value:.4f}")


# ── 3. Final LOCO evaluation with PCA and scaling inside each fold ────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

print("═════════════════════════════════════ GPR with PCA ═════════════════════════════════════\n")

# PCA definitions (same as in objective, but defined once for the LOCO loop)
pca_cat_final  = PCA(n_components=4)
pca_mol_final  = PCA(n_components=4)
pca_rf_final   = PCA(n_components=12)

GROUP_CFG_FINAL = [
    ('cat',  pca_cat_final,  cat_feats),
    ('mol',  pca_mol_final,  mol_descps),
    ('rf',   pca_rf_final,   rf_cols),
]
all_num_cols_final = cat_feats + mol_descps + rf_cols


for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    pca_tr_parts  = []
    pca_val_parts = []

    for prefix, pca_obj, cols in GROUP_CFG_FINAL:
        scaler     = StandardScaler()
        tr_scaled  = scaler.fit_transform(X_train[cols])
        val_scaled = scaler.transform(X_test[cols])

        tr_pca  = pca_obj.fit_transform(tr_scaled)
        val_pca = pca_obj.transform(val_scaled)

        pca_col_names = [f'{prefix}_pc{i}' for i in range(pca_obj.n_components_)]
        pca_tr_parts.append(
            pd.DataFrame(tr_pca,  columns=pca_col_names, index=X_train.index))
        pca_val_parts.append(
            pd.DataFrame(val_pca, columns=pca_col_names, index=X_test.index))

    other_cols_train = X_train.drop(columns=all_num_cols_final, errors='ignore')
    X_train_processed  = pd.concat([other_cols_train, *pca_tr_parts], axis=1)

    other_cols_test = X_test.drop(columns=all_num_cols_final, errors='ignore')
    X_test_processed = pd.concat([other_cols_test, *pca_val_parts], axis=1)

    model = GaussianProcessRegressor(
        kernel=GPR_KERNEL,
        alpha=1e-6,
        normalize_y=True,
        n_restarts_optimizer=gpr_best_params.get('n_restarts_optimizer', 5),
        random_state=42
    )

    model.fit(X_train_processed, y_train.values.ravel())
    preds = model.predict(X_test_processed)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")


# ── 4. Summary ────────────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")


  0%|          | 0/80 [00:00<?, ?it/s]

Best params : {'n_restarts_optimizer': 10}
Best CV R²  : 0.6873
═════════════════════════════════════ GPR with PCA ═════════════════════════════════════

Held-out: Cu              R²=0.5608  MAE=0.1844  RMSE=0.2514  MAPE=60.51%
Held-out: Ni              R²=0.6279  MAE=0.1997  RMSE=0.2344  MAPE=inf%
Held-out: Pd              R²=0.7888  MAE=0.1170  RMSE=0.1690  MAPE=44.44%
Held-out: Pt              R²=0.6985  MAE=0.1790  RMSE=0.2335  MAPE=115.43%
Held-out: Rh              R²=0.7605  MAE=0.1336  RMSE=0.1772  MAPE=inf%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.6873     0.0840     0.5608     0.7888
MAE          0.1628     0.0317     0.1170     0.1997
RMSE         0.2131     0.0334     0.1690     0.2514
MSE          0.0465     0.0139     0.0286     0.0632
MAPE%           inf        nan      44.44        inf


# ══════════════ **STACKING** ══════════════

In [ ]:
# ── 1. Use already tuned params from each model ───────────────────────────────
# ── 2. Final LOCO evaluation ──────────────────────────────────────────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

print("═════════════════════════════════════ Stacking ═════════════════════════════════════\n")

for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    scaler = RobustScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols]  = scaler.transform(X_test[num_cols])

    stack = StackingRegressor(
        estimators=[
            #('rf',   RandomForestRegressor(**rf_best_params,   random_state=42, n_jobs=-1)),
            ('lgbm', LGBMRegressor(**lgbm_best_params,         random_state=42, verbose=-1, n_jobs=-1)),
            ('xgb',  XGBRegressor(**xgb_best_params,           random_state=42, verbosity=0, n_jobs=-1)),
            #('svr',  SVR(**svr_best_params)),
            ('gpr',  GaussianProcessRegressor(
                         kernel=GPR_KERNEL, alpha=1e-6,
                         normalize_y=True, n_restarts_optimizer=3,
                         random_state=42)),
        ],
        final_estimator = Ridge(alpha=1.0),
        cv              = 5,
        n_jobs          = -1
    )

    stack.fit(X_train, y_train)
    preds  = stack.predict(X_test)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")


# ── 3. Summary ────────────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")

═════════════════════════════════════ Stacking ═════════════════════════════════════

Held-out: Cu              R²=0.6281  MAE=0.1607  RMSE=0.2313  MAPE=56.30%
Held-out: Ni              R²=0.6673  MAE=0.1907  RMSE=0.2216  MAPE=inf%
Held-out: Pd              R²=0.8308  MAE=0.1084  RMSE=0.1513  MAPE=46.07%
Held-out: Pt              R²=0.7174  MAE=0.1726  RMSE=0.2261  MAPE=119.12%
Held-out: Rh              R²=0.7510  MAE=0.1432  RMSE=0.1807  MAPE=inf%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.7189     0.0699     0.6281     0.8308
MAE          0.1551     0.0281     0.1084     0.1907
RMSE         0.2022     0.0311     0.1513     0.2313
MSE          0.0419     0.0120     0.0229     0.0535
MAPE%           inf        nan      46.07        inf


In [ ]:
# ── 1. Use already tuned params from each model ───────────────────────────────
# ── 2. Final LOCO evaluation ──────────────────────────────────────────────────
logo       = LeaveOneGroupOut()
loco_r2    = []
loco_preds = {}

print("═════════════════════════════════════ Stacking ═════════════════════════════════════\n")

pca_cat  = PCA(n_components=4)   # catalyst properties
pca_mol  = PCA(n_components=4)   # molecular descriptors
pca_rf   = PCA(n_components=12)   # reactant + product atom count

GROUP_CFG = [
    ('cat',  pca_cat,  cat_feats),
    ('mol',  pca_mol,  mol_descps),
    ('rf',   pca_rf,   rf_cols),
]
all_num_cols = cat_feats + mol_descps + rf_cols

for train_idx, test_idx in logo.split(X, y, groups):
    X_train = X.iloc[train_idx].copy()
    X_test  = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]

    held_out_catalyst = np.unique(groups[test_idx])[0]

    pca_tr_parts  = []
    pca_val_parts = []

    for prefix, pca_obj, cols in GROUP_CFG:
        scaler     = StandardScaler()
        tr_scaled  = scaler.fit_transform(X_train[cols])
        val_scaled = scaler.transform(X_test[cols])

        tr_pca  = pca_obj.fit_transform(tr_scaled)
        val_pca = pca_obj.transform(val_scaled)

        pca_col_names = [f'{prefix}_pc{i}' for i in range(pca_obj.n_components_)]
        pca_tr_parts.append(
            pd.DataFrame(tr_pca,  columns=pca_col_names, index=X_train.index))
        pca_val_parts.append(
            pd.DataFrame(val_pca, columns=pca_col_names, index=X_test.index))

    other_cols_train = X_train.drop(columns=all_num_cols, errors='ignore')
    X_train_processed  = pd.concat([other_cols_train, *pca_tr_parts], axis=1)

    other_cols_test = X_test.drop(columns=all_num_cols, errors='ignore')
    X_test_processed = pd.concat([other_cols_test, *pca_val_parts], axis=1)

    stack = StackingRegressor(
        estimators=[
            #('rf',   RandomForestRegressor(**rf_best_params,   random_state=42, n_jobs=-1)),
            ('lgbm', LGBMRegressor(**lgbm_best_params,         random_state=42, verbose=-1, n_jobs=-1)),
            ('xgb',  XGBRegressor(**xgb_best_params,           random_state=42, verbosity=0, n_jobs=-1)),
            #('svr',  SVR(**svr_best_params)),
            ('gpr',  GaussianProcessRegressor(
                         kernel=GPR_KERNEL, alpha=1e-6,
                         normalize_y=True, n_restarts_optimizer=3,
                         random_state=42)),
        ],
        final_estimator = Ridge(alpha=1.0),
        cv              = 5,
        n_jobs          = -1
    )

    stack.fit(X_train_processed, y_train)
    preds  = stack.predict(X_test_processed)
    y_true = y_test.values

    r2   = r2_score(y_true, preds)
    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mse  = mean_squared_error(y_true, preds)
    mape = np.mean(np.abs((y_true - preds) / y_true)) * 100

    loco_r2.append(r2)
    loco_preds[held_out_catalyst] = {
        'y_true': y_true, 'y_pred': preds,
        'r2': r2, 'mae': mae, 'rmse': rmse, 'mse': mse, 'mape': mape
    }
    print(f"Held-out: {held_out_catalyst:<15} R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")


# ── 3. Summary ────────────────────────────────────────────────────────────────
r2s   = [v['r2']   for v in loco_preds.values()]
maes  = [v['mae']  for v in loco_preds.values()]
rmses = [v['rmse'] for v in loco_preds.values()]
mses  = [v['mse']  for v in loco_preds.values()]
mapes = [v['mape'] for v in loco_preds.values()]

print(f"\n{'Metric':<8} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 50)
print(f"{'R²':<8} {np.mean(r2s):>10.4f} {np.std(r2s):>10.4f} {np.min(r2s):>10.4f} {np.max(r2s):>10.4f}")
print(f"{'MAE':<8} {np.mean(maes):>10.4f} {np.std(maes):>10.4f} {np.min(maes):>10.4f} {np.max(maes):>10.4f}")
print(f"{'RMSE':<8} {np.mean(rmses):>10.4f} {np.std(rmses):>10.4f} {np.min(rmses):>10.4f} {np.max(rmses):>10.4f}")
print(f"{'MSE':<8} {np.mean(mses):>10.4f} {np.std(mses):>10.4f} {np.min(mses):>10.4f} {np.max(mses):>10.4f}")
print(f"{'MAPE%':<8} {np.mean(mapes):>10.2f} {np.std(mapes):>10.2f} {np.min(mapes):>10.2f} {np.max(mapes):>10.2f}")

═════════════════════════════════════ Stacking ═════════════════════════════════════

Held-out: Cu              R²=0.6369  MAE=0.1607  RMSE=0.2286  MAPE=59.74%
Held-out: Ni              R²=0.6665  MAE=0.1856  RMSE=0.2219  MAPE=inf%
Held-out: Pd              R²=0.8433  MAE=0.1033  RMSE=0.1456  MAPE=45.92%
Held-out: Pt              R²=0.7166  MAE=0.1743  RMSE=0.2264  MAPE=118.22%
Held-out: Rh              R²=0.7664  MAE=0.1308  RMSE=0.1750  MAPE=inf%

Metric         Mean        Std        Min        Max
--------------------------------------------------
R²           0.7259     0.0734     0.6369     0.8433
MAE          0.1509     0.0300     0.1033     0.1856
RMSE         0.1995     0.0334     0.1456     0.2286
MSE          0.0409     0.0126     0.0212     0.0523
MAPE%           inf        nan      45.92        inf
